# KG1 V1243 Colab Realtime Launcher

Colab URL:

`https://colab.research.google.com/github/FELIPEACASTRO/KG1-NVIDIA/blob/master/notebooks/KG1_V1243_COLAB_REALTIME_LAUNCHER.ipynb`

Purpose: open the V1243 launch pack, stream logs to a private Hugging Face dataset, and run the safe launch sequence with Kaggle submission disabled.

Default behavior:

- upload or locate `v1243_colab_launch_pack.zip`;
- unpack it under `/content/kg1_v1243`;
- install dependencies;
- run the tokenization dry run first;
- leave GPU model-load and real run behind explicit environment flags.


In [ ]:
# CELL: configure runtime, secrets, and hard locks.
print('=== V1243 REALTIME CONFIG START ===', flush=True)
import datetime
import json
import os
import pathlib
import subprocess
import sys
import time
import zipfile

ALLOW_KAGGLE_SUBMIT = False
if ALLOW_KAGGLE_SUBMIT:
    raise RuntimeError('Kaggle submission is disabled in this notebook.')

COLAB_URL = 'https://colab.research.google.com/github/FELIPEACASTRO/KG1-NVIDIA/blob/master/notebooks/KG1_V1243_COLAB_REALTIME_LAUNCHER.ipynb'
ROOT = pathlib.Path('/content/kg1_v1243')
PACK_ZIP_NAME = 'v1243_colab_launch_pack.zip'
LOG_ROOT = pathlib.Path('/content/kg1_live_logs')
LOG_ROOT.mkdir(parents=True, exist_ok=True)
PHASE = os.environ.get('KG1_V1243_PHASE', 'bit_specialist')
TARGET_ACCURACY = os.environ.get('KG1_TARGET_ACCURACY', '0.98')
RUN_MODEL_DRYRUN = os.environ.get('KG1_V1243_RUN_MODEL_DRYRUN', '0')
RUN_TRAIN = os.environ.get('KG1_V1243_RUN_TRAIN', '0')
repo_commit = os.environ.get('KG1_REPO_COMMIT', 'local-launch-pack')

def read_colab_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

if not os.environ.get('HF_TOKEN'):
    token = read_colab_secret('HF_TOKEN') or read_colab_secret('HUGGINGFACE_TOKEN') or read_colab_secret('HF_KEY')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGINGFACE_HUB_TOKEN'] = token

os.environ.setdefault('PYTHONUNBUFFERED', '1')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('FRIENDLY_REALTIME_LOGS', '1')
os.environ.setdefault('FRIENDLY_LOG_SCORE_HINTS', '1')
os.environ.setdefault('KG1_LIVE_LOG_HF_REPO', 'felipesp1983/kg1-live-logs')
os.environ.setdefault('KG1_LIVE_LOG_HF_REPO_TYPE', 'dataset')

print('colab_url =', COLAB_URL, flush=True)
print('repo_commit =', repo_commit, flush=True)
print('phase =', PHASE, flush=True)
print('target_accuracy =', TARGET_ACCURACY, flush=True)
print('hf_token_ready =', bool(os.environ.get('HF_TOKEN')), flush=True)
print('live_log_repo =', os.environ.get('KG1_LIVE_LOG_HF_REPO'), flush=True)
print('run_model_dryrun =', RUN_MODEL_DRYRUN, flush=True)
print('run_train =', RUN_TRAIN, flush=True)
print('allow_kaggle_submit =', ALLOW_KAGGLE_SUBMIT, flush=True)
print('=== V1243 REALTIME CONFIG END ===', flush=True)


In [ ]:
# CELL: upload or locate the launch pack.
print('=== V1243 PACK SETUP START ===', flush=True)
from pathlib import Path

ROOT.mkdir(parents=True, exist_ok=True)
zip_path = Path('/content') / PACK_ZIP_NAME
if not zip_path.exists():
    print('launch_pack_zip_missing=True; opening Colab upload dialog', flush=True)
    from google.colab import files
    uploaded = files.upload()
    for name, payload in uploaded.items():
        candidate = Path('/content') / name
        candidate.write_bytes(payload)
        print('uploaded_file =', candidate, 'bytes =', candidate.stat().st_size, flush=True)
        if name.endswith('.zip'):
            zip_path = candidate
            break
if not zip_path.exists():
    raise FileNotFoundError('Upload v1243_colab_launch_pack.zip before continuing.')
print('launch_pack_zip =', zip_path, 'bytes =', zip_path.stat().st_size, flush=True)
with zipfile.ZipFile(zip_path) as archive:
    members = archive.namelist()
    print('zip_members =', len(members), flush=True)
    archive.extractall(ROOT)
print('pack_root =', ROOT, flush=True)
print('pack_manifest_exists =', (ROOT / 'kg1_v1243_colab_launch_pack_manifest.json').exists(), flush=True)
print('=== V1243 PACK SETUP END ===', flush=True)


In [ ]:
# CELL: define command logger and install dependencies.
print('=== V1243 DEPENDENCIES START ===', flush=True)
import py_compile

def run_cmd(cmd, *, cwd=None, log_path=None, check=True):
    cwd = pathlib.Path(cwd or ROOT)
    log_path = pathlib.Path(log_path or (LOG_ROOT / ('cmd_' + str(int(time.time())) + '.log')))
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('COMMAND START', json.dumps({'cmd': [str(x) for x in cmd], 'cwd': str(cwd), 'log_path': str(log_path)}), flush=True)
    with log_path.open('w', encoding='utf-8', buffering=1) as log:
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        tail = []
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
            tail.append(line.rstrip())
            if len(tail) > 30:
                tail.pop(0)
        returncode = proc.wait()
    print('COMMAND END', json.dumps({'returncode': returncode, 'log_path': str(log_path)}), flush=True)
    if check and returncode != 0:
        print('command_tail_on_failure =', '\n'.join(tail[-20:]), flush=True)
        raise RuntimeError('command failed with returncode=' + str(returncode))
    return returncode

for rel in [
    'scripts/kg1_colab_v1243_launcher.py',
    'scripts/kg1_colab_realtime_runner.py',
    'scripts/kg1_colab_live_monitor.py',
    'scripts/kg1_live_log_common.py',
]:
    py_compile.compile(str(ROOT / rel), doraise=True)
print('py_compile = PASS', flush=True)
run_cmd(
    [sys.executable, '-m', 'pip', 'install', '-q', '-U', '-r', str(ROOT / 'requirements_v1243_colab.txt')],
    cwd=ROOT,
    log_path=LOG_ROOT / 'requirements_install.log',
)
print('=== V1243 DEPENDENCIES END ===', flush=True)


In [ ]:
# CELL: run tokenization dry run with live logs.
print('=== V1243 TOKENIZE DRYRUN START ===', flush=True)
RUN_ID = 'v1243_' + PHASE + '_tokenize_' + time.strftime('%Y%m%d_%H%M%S')
os.environ['RUN_ID'] = RUN_ID
os.environ['KG1_LIVE_LOG_HF_PATH'] = 'colab/' + RUN_ID + '/train.log'
os.environ['KG1_LIVE_STATUS_HF_PATH'] = 'colab/' + RUN_ID + '/status.json'
print('run_id =', RUN_ID, flush=True)
print('live_log_path =', os.environ['KG1_LIVE_LOG_HF_PATH'], flush=True)
run_cmd(
    [
        sys.executable,
        'scripts/kg1_colab_v1243_launcher.py',
        '--phase', PHASE,
        '--run-mode', 'tokenize_dryrun',
        '--target-accuracy', TARGET_ACCURACY,
        '--live-log-repo', os.environ.get('KG1_LIVE_LOG_HF_REPO', ''),
        '--live-log-repo-type', os.environ.get('KG1_LIVE_LOG_HF_REPO_TYPE', 'dataset'),
    ],
    cwd=ROOT,
    log_path=LOG_ROOT / (RUN_ID + '_launcher.log'),
)
print('monitor_command = python scripts\\kg1_colab_live_monitor.py --hf-repo ' + os.environ.get('KG1_LIVE_LOG_HF_REPO', '') + ' --hf-path ' + os.environ['KG1_LIVE_LOG_HF_PATH'] + ' --hf-repo-type dataset --interval 30 --target-accuracy ' + TARGET_ACCURACY, flush=True)
print('=== V1243 TOKENIZE DRYRUN END ===', flush=True)


In [ ]:
# CELL: optional GPU model-load dry run.
print('=== V1243 MODEL DRYRUN START ===', flush=True)
if RUN_MODEL_DRYRUN != '1':
    print('model_dryrun_skipped=True set KG1_V1243_RUN_MODEL_DRYRUN=1 and rerun this cell when ready', flush=True)
else:
    RUN_ID = 'v1243_' + PHASE + '_modeldry_' + time.strftime('%Y%m%d_%H%M%S')
    os.environ['RUN_ID'] = RUN_ID
    os.environ['KG1_LIVE_LOG_HF_PATH'] = 'colab/' + RUN_ID + '/train.log'
    os.environ['KG1_LIVE_STATUS_HF_PATH'] = 'colab/' + RUN_ID + '/status.json'
    print('run_id =', RUN_ID, flush=True)
    run_cmd(
        [
            sys.executable,
            'scripts/kg1_colab_v1243_launcher.py',
            '--phase', PHASE,
            '--run-mode', 'model_dryrun',
            '--target-accuracy', TARGET_ACCURACY,
            '--live-log-repo', os.environ.get('KG1_LIVE_LOG_HF_REPO', ''),
            '--live-log-repo-type', os.environ.get('KG1_LIVE_LOG_HF_REPO_TYPE', 'dataset'),
        ],
        cwd=ROOT,
        log_path=LOG_ROOT / (RUN_ID + '_launcher.log'),
    )
print('=== V1243 MODEL DRYRUN END ===', flush=True)


In [ ]:
# CELL: optional real run, hard gated.
print('=== V1243 REAL RUN START ===', flush=True)
if RUN_TRAIN != '1':
    print('real_run_skipped=True set KG1_V1243_RUN_TRAIN=1 only after dry runs pass', flush=True)
else:
    output_repo = os.environ.get('OUTPUT_REPO', '')
    if not output_repo:
        raise RuntimeError('OUTPUT_REPO is required for real run.')
    RUN_ID = 'v1243_' + PHASE + '_real_' + time.strftime('%Y%m%d_%H%M%S')
    os.environ['RUN_ID'] = RUN_ID
    os.environ['KG1_LIVE_LOG_HF_PATH'] = 'colab/' + RUN_ID + '/train.log'
    os.environ['KG1_LIVE_STATUS_HF_PATH'] = 'colab/' + RUN_ID + '/status.json'
    print('run_id =', RUN_ID, flush=True)
    run_cmd(
        [
            sys.executable,
            'scripts/kg1_colab_v1243_launcher.py',
            '--phase', PHASE,
            '--run-mode', 'real_train',
            '--allow-real-train',
            '--target-accuracy', TARGET_ACCURACY,
            '--live-log-repo', os.environ.get('KG1_LIVE_LOG_HF_REPO', ''),
            '--live-log-repo-type', os.environ.get('KG1_LIVE_LOG_HF_REPO_TYPE', 'dataset'),
            '--output-repo', output_repo,
        ],
        cwd=ROOT,
        log_path=LOG_ROOT / (RUN_ID + '_launcher.log'),
    )
print('=== V1243 REAL RUN END ===', flush=True)
